# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [ ]:
q1 = q('''
SELECT
  t.title,
  a.name AS artist_name,
  a.country
FROM
  tracks AS t
JOIN
  artists AS a
ON
  t.artist_id = a.artist_id;
''')
q1

,title,artist_name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


In my code above I create a SQL query to create a list of all the tracks, the artist's name, and artist's country. It creates the titles from the tracks table, the name from the artists table, and then renames it to the artist_name, and then incorporates the country from the artists table.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [ ]:
q2 = q('''
SELECT
  genre,
  AVG(seconds) AS average_track_length
FROM
  tracks
WHERE
  genre IS NOT NULL
GROUP BY
  genre
ORDER BY
  average_track_length DESC
LIMIT 1;
''')
q2

,genre,average_track_length
0,Electronic,287.5


In my code above it is calculating the length by selecting a specific genre and then calculating the averages for length. It filters for null values, so where a genre is selected and it is null it is not included in this calculation.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [ ]:
q3 = q('''
SELECT
  user,
  COUNT(play_id) AS total_plays,
  COUNT(DISTINCT track_id) AS distinct_tracks
FROM
  plays
GROUP BY
  user;
''')
q3

,user,total_plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


My code above makes clear in the results to differentiate between an individual who played one track four times and someone who played four different tracks by having designated columns.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [ ]:
q4 = q('''
SELECT
  t.track_id,
  t.title
FROM
  tracks AS t
LEFT JOIN
  plays AS p
ON
  t.track_id = p.track_id
WHERE
  p.play_id IS NULL;
''')
q4

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


I started by including all of the tracks from the tracks table, serving as the "left" table. Then I tried to match each track from the track table with its connected plays from the plays table. The ones that have been played will automatically match with the their associated track from the plays table. But the ones that haven't been played will show no match. But because of the LEFT JOIN it still includes the tracks from the tracks table and then the columns from the plays table will have null for that specific track. Where a null result is shown we know that they have zero plays and they get printed.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [ ]:
q5 = q('''
SELECT
  ar.name AS artist_name,
  SUM(t.seconds) AS total_seconds_listened,
  ROUND(SUM(t.seconds) / 60.0, 1) AS total_minutes_listened
FROM
  plays AS p
JOIN
  tracks AS t
ON
  p.track_id = t.track_id
JOIN
  artists AS ar
ON
  t.artist_id = ar.artist_id
GROUP BY
  ar.name
ORDER BY
  total_seconds_listened DESC;
''')
q5

,artist_name,total_seconds_listened,total_minutes_listened
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


In my code above it takes each play, and sees what specific song from the 'Song list' it was so it can see how long the song was. Following that for the same songs it then looks up the artist from the 'artists list' to get that name. After they all get grouped accordigly. After the times get converted and then each artist is shown with their total times, and arranges from highest to lowest.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [ ]:
q6 = q('''
SELECT
  track_id,
  title
FROM
  tracks
WHERE
  genre IS NULL;
''')
q6

,track_id,title
0,18,Untitled Demo


If I used WHERE genre!= 'Pop' instead of WHERE genre IS NULL when filter the genres, the tracks with NULL genre would not be included in the final results.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [ ]:
q7 = q('''
SELECT
  played_on AS play_date,
  COUNT(play_id) AS total_plays,
  COUNT(DISTINCT user) AS distinct_users
FROM
  plays
GROUP BY
  played_on
ORDER BY
  played_on ASC;
''')
q7

,play_date,total_plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


My code above analyzes the times song were palyed and then looks at the total amounts it was played on a given day and how many different people listened to the song on that specific day. It then all gets sorted respectively to produce the table.  

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [ ]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['total_plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

The query that gave me the most trouble was Query 4. This was because I had originally tried to do it using a INNER JOIN over a LEFT JOIN while figuring out which records don't have matches. If I had stuck with the inner join I would've only gotten tracks that do have at least one play because I realized inner join will produce rows when there is a match in both of the tables. So it is good that I changed it to LEFT JOIN because otherwise it would have accidently not included the tracks that had no connected entries in the plays table.